# BACE1 (GC4-D3R) — kNN modular por funciones (clasificación y regresión) sobre matrices de distancia de MEP

**Licencia:** MIT (ver archivo `LICENSE` en la raíz del repositorio).

Este cuaderno es la versión "modular" del pipeline: encapsula la clasificación y la
regresión kNN en funciones reutilizables (`perform_knn_classification` y
`perform_knn_regression`), aplicadas a cada una de las cinco combinaciones de matrices de
distancia derivadas del Potencial Electrostático Molecular (MEP): negativo puro, positivo
puro, y mezclas 50-50, 30-70 y 70-30.

Ver `notebooks/01_KNN_clasificacion_regresion.ipynb` para la versión con barrido de K
(método del codo) y dendrogramas coloreados; este cuaderno usa un K fijo (`n_neighbors=5`
por defecto) y está pensado como reporte final por matriz.

**Datos:** igual que en el cuaderno 01 — 153 moléculas conocidas usadas para
entrenar/evaluar (`mol116` queda fuera porque no tiene MEP calculable) y 20 moléculas de
pose (`xyz_newmol0`...`xyz_newmol19`) sin afinidad experimental, para las que se predice
clase y valor de afinidad.

## 0. Configuración de entorno

In [1]:
# Bandera para decidir si se monta Google Drive (Colab) o se usa una carpeta local del repo.
USAR_DRIVE = False  # cambiar a True si se ejecuta en Colab con los datos en Drive

if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/semillero/qcamat/Distance_matrix_bace'
else:
    RUTA_BASE = '../data'

In [2]:
import pandas as pd
import numpy as np

## 1. Carga de la tabla de afinidades

Misma corrección que en el cuaderno 01: `ID_real` en el CSV viene como entero plano
(`0`, `1`, ...), se normaliza a `"molN"` para que coincida con las etiquetas de las
matrices de distancia.

In [3]:
afinidad_csv_path = f'{RUTA_BASE}/BACE_affinity.csv'
bace_affinity_df = pd.read_csv(afinidad_csv_path, sep=';')

def normalizar_id_real(valor):
    """Devuelve el ID_real en formato 'molN', sin duplicar el prefijo si ya viene incluido."""
    texto = str(valor).strip()
    return texto if texto.startswith('mol') else f'mol{texto}'

bace_affinity_df['ID_real'] = bace_affinity_df['ID_real'].apply(normalizar_id_real)

print(f"Filas cargadas: {len(bace_affinity_df)}")
print(bace_affinity_df.head())

Filas cargadas: 154
       ID                                             SMILES  Affinity ID_real
0  BACE_1  CCCCNC(=O)[C@H](C)C[C@H](O)[C@@H]2C[C@H](C)CCC...    0.0160    mol0
1  BACE_4  C[C@@H]1CCCCCCOCC(=O)N(C)[C@@H](C)C(=O)N[C@@H]...    0.0075    mol1
2  BACE_5  CCCC2=CC(CNC[C@@H](O)[C@@H]1C[C@H](C)CCCCCCOCC...    0.4000    mol2
3  BACE_6  CCN2CCCCC[C@@H](C)C[C@H](NC(=O)c1cc(ccc1)C2=O)...    0.0210    mol3
4  BACE_7  CC(C)c5cc(CNC[C@@H](O)[C@@H]4C[C@H](C)CCCCCN([...    0.0020    mol4


## 2. Función de color por rango de afinidad

Misma corrección de bins contiguos que en el cuaderno 01 (los rangos originales dejaban
huecos entre categorías vecinas, lo que podía mandar predicciones a un color "negro por
defecto" no documentado).

In [4]:
def get_affinity_color(affinity):
    """Asigna un color/categoría según el valor de afinidad, con límites contiguos."""
    if affinity > 8.9:
        return '#008000'   # Verde oliva oscuro: afinidad alta (8.9 < A)
    elif affinity > 0.96:
        return '#8B7500'   # Amarillo mostaza / dorado oscuro (0.96 < A <= 8.9)
    elif affinity > 0.093:
        return '#8B0000'   # Rojo vino / carmesí (0.093 < A <= 0.96)
    elif affinity > 0.0095:
        return '#555555'   # Gris oscuro neutro (0.0095 < A <= 0.093)
    else:
        return '#0000CD'   # Azul rey / azul medio (A <= 0.0095)

color_hex_to_name_map = {
    '#008000': 'DarkGreen',
    '#8B7500': 'MustardYellow',
    '#8B0000': 'DarkRed',
    '#555555': 'DarkGray',
    '#0000CD': 'MediumBlue',
}

color_map = {
    row['ID_real']: get_affinity_color(row['Affinity'])
    for _, row in bace_affinity_df.iterrows()
}

print(f"Mapa de colores creado para {len(color_map)} moléculas conocidas.")

Mapa de colores creado para 154 moléculas conocidas.


## 3. Carga y limpieza de las matrices de distancia

**Corrección crítica (igual que en el cuaderno 01):** `clean_mol_id` original colapsaba
`xyz_newmol0`...`xyz_newmol19` a un único label duplicado `"xyz"`, lo que rompía la
indexación por `.loc[]` de las moléculas de pose (el cuaderno original evitaba el problema
a medias usando slicing posicional `.iloc[]`, una solución fràgil que dependía de que el
orden de filas no cambiara). Aquí se corrige de raíz conservando el identificador completo
y único para las moléculas de pose, y el resto del cuaderno vuelve a poder indexar por
`.loc[]` de forma directa y explícita.

In [5]:
distance_neg = pd.read_csv(f'{RUTA_BASE}/distance_matrix_neg', header=None, sep=',')
distance_pos_col = pd.read_csv(f'{RUTA_BASE}/distance_matrix_pos_col', header=None, sep=',')
distance_mix_50_50 = pd.read_csv(f'{RUTA_BASE}/mix_50_50.csv', header=None, sep=',')
distance_mix_30_70 = pd.read_csv(f'{RUTA_BASE}/mix_30_70.csv', header=None, sep=',')
distance_mix_70_30 = pd.read_csv(f'{RUTA_BASE}/mix_70_30.csv', header=None, sep=',')

In [6]:
raw_mol_ids_from_matrix = distance_neg.iloc[0, 1:].tolist()

def clean_mol_id(mol_id_str):
    """
    'molN_...' -> 'molN'. 'xyz_newmolN' se conserva completo y único
    (ver nota de corrección en la celda anterior).
    """
    texto = str(mol_id_str)
    prefijo = texto.split('_')[0]
    return prefijo if prefijo.startswith('mol') else texto

mol_ids_from_matrix_cleaned = [clean_mol_id(x) for x in raw_mol_ids_from_matrix]
print(f"Moléculas esperadas según distance_neg: {len(mol_ids_from_matrix_cleaned)}")
print(f"Identificadores únicos: {len(set(mol_ids_from_matrix_cleaned))}")

def normalize_numeric_df(df_numeric):
    """Normaliza un DataFrame numérico al rango [0, 1] con min/max globales."""
    global_min = df_numeric.values.min()
    global_max = df_numeric.values.max()
    if (global_max - global_min) == 0:
        return df_numeric - global_min
    return (df_numeric - global_min) / (global_max - global_min)

distance_neg_numeric = distance_neg.iloc[1:, 1:].astype(float)
distance_neg_numeric.columns = mol_ids_from_matrix_cleaned
distance_neg_numeric.index = mol_ids_from_matrix_cleaned
distance_neg_norm = normalize_numeric_df(distance_neg_numeric)
print(f"Forma de distance_neg_norm: {distance_neg_norm.shape}")

distance_pos_col_numeric = distance_pos_col.iloc[1:, 1:].astype(float)
distance_pos_col_numeric.columns = mol_ids_from_matrix_cleaned
distance_pos_col_numeric.index = mol_ids_from_matrix_cleaned
distance_pos_col_norm = normalize_numeric_df(distance_pos_col_numeric)
print(f"Forma de distance_pos_col_norm: {distance_pos_col_norm.shape}")

distance_mix_50_50_numeric = distance_mix_50_50.astype(float)
distance_mix_50_50_numeric.columns = mol_ids_from_matrix_cleaned
distance_mix_50_50_numeric.index = mol_ids_from_matrix_cleaned
distance_mix_50_50_norm = normalize_numeric_df(distance_mix_50_50_numeric)
print(f"Forma de distance_mix_50_50_norm: {distance_mix_50_50_norm.shape}")

distance_mix_30_70_numeric = distance_mix_30_70.astype(float)
distance_mix_30_70_numeric.columns = mol_ids_from_matrix_cleaned
distance_mix_30_70_numeric.index = mol_ids_from_matrix_cleaned
distance_mix_30_70_norm = normalize_numeric_df(distance_mix_30_70_numeric)
print(f"Forma de distance_mix_30_70_norm: {distance_mix_30_70_norm.shape}")

distance_mix_70_30_numeric = distance_mix_70_30.astype(float)
distance_mix_70_30_numeric.columns = mol_ids_from_matrix_cleaned
distance_mix_70_30_numeric.index = mol_ids_from_matrix_cleaned
distance_mix_70_30_norm = normalize_numeric_df(distance_mix_70_30_numeric)
print(f"Forma de distance_mix_70_30_norm: {distance_mix_70_30_norm.shape}")

print("Todas las matrices de distancia fueron limpiadas, indexadas y normalizadas.")

Moléculas esperadas según distance_neg: 173
Identificadores únicos: 173
Forma de distance_neg_norm: (173, 173)
Forma de distance_pos_col_norm: (173, 173)
Forma de distance_mix_50_50_norm: (173, 173)
Forma de distance_mix_30_70_norm: (173, 173)
Forma de distance_mix_70_30_norm: (173, 173)
Todas las matrices de distancia fueron limpiadas, indexadas y normalizadas.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              mean_squared_error, r2_score)
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 4. K-Nearest Neighbors (kNN) — Clasificación

Se usa `KNeighborsClassifier` con `metric='precomputed'`, entrenado sobre las moléculas
conocidas y evaluado con matriz de confusión y reporte de clasificación. Luego el modelo
entrenado se usa para predecir la categoría de afinidad de las 20 moléculas de pose.

Primero separamos las moléculas en **conocidas** (con afinidad en `bace_affinity_df`) y
**desconocidas** (presentes en las matrices de distancia pero sin afinidad), y preparamos
`y_known_colors` para las conocidas.

In [8]:
# 1. Extraer IDs de moléculas conocidas y desconocidas.
known_mol_ids_from_affinity_df = bace_affinity_df['ID_real'].astype(str).tolist()
all_mol_ids_in_matrices = mol_ids_from_matrix_cleaned

# Moléculas con afinidad conocida que además están presentes en las matrices de distancia
# (esto excluye automáticamente a 'mol116', que no tiene MEP calculable).
known_mol_ids_present_in_matrix = [
    mol_id for mol_id in known_mol_ids_from_affinity_df
    if mol_id in all_mol_ids_in_matrices
]
y_known_colors_present_in_matrix = [color_map[mol_id] for mol_id in known_mol_ids_present_in_matrix]

# Moléculas realmente desconocidas: están en las matrices pero no tienen afinidad.
# Con la corrección de clean_mol_id, estos 20 IDs ya son únicos ('xyz_newmol0', ...),
# en vez de 20 copias duplicadas de 'xyz'.
unknown_mol_ids_list = [
    mol_id for mol_id in all_mol_ids_in_matrices
    if mol_id not in known_mol_ids_present_in_matrix
]

print(f"Moléculas con afinidad (inicial): {len(known_mol_ids_from_affinity_df)}")
print(f"Moléculas conocidas presentes en las matrices: {len(known_mol_ids_present_in_matrix)}")
print(f"Moléculas desconocidas (en matrices, sin afinidad): {len(unknown_mol_ids_list)}")
print(f"Total de moléculas en las matrices: {len(all_mol_ids_in_matrices)}")

known_mol_ids_list = known_mol_ids_present_in_matrix
y_known_colors = y_known_colors_present_in_matrix

Moléculas con afinidad (inicial): 154
Moléculas conocidas presentes en las matrices: 153
Moléculas desconocidas (en matrices, sin afinidad): 20
Total de moléculas en las matrices: 173


Función `perform_knn_classification`:
1. Separa la sub-matriz de distancias entre moléculas conocidas.
2. Divide los **índices** de las moléculas conocidas en entrenamiento/prueba (`stratify` por
   color, para mantener la proporción de clases).
3. Entrena un `KNeighborsClassifier` con `metric='precomputed'`.
4. Evalúa sobre el conjunto de prueba (matriz de confusión + reporte de clasificación).
5. Predice la categoría de afinidad para las moléculas desconocidas.

**Corrección:** la extracción de las distancias "desconocidas -> entrenamiento" ahora usa
`.loc[unknown_mol_ids, train_mol_ids]` (indexación por etiqueta, clara y explícita) en vez
del slicing posicional `.iloc[num_known : num_known + num_unknown]` del cuaderno original,
que dependía silenciosamente de que las 20 moléculas desconocidas estuvieran siempre al
final y en el mismo orden dentro de la matriz.

In [9]:
def perform_knn_classification(full_distance_df, matrix_name, known_mol_ids, unknown_mol_ids,
                                y_known_labels, n_neighbors=5):
    print(f"\n--- Clasificando con {matrix_name} ---\n")

    # 1. Sub-matriz de distancias entre moléculas conocidas.
    X_known_full = full_distance_df.loc[known_mol_ids, known_mol_ids]

    # 2. División train/test sobre los índices de las moléculas conocidas.
    known_indices = list(range(len(known_mol_ids)))
    train_idx, test_idx, y_train_labels_split, y_test_labels_split = train_test_split(
        known_indices, y_known_labels, test_size=0.2, random_state=42, stratify=y_known_labels
    )
    train_mol_ids = [known_mol_ids[i] for i in train_idx]
    test_mol_ids = [known_mol_ids[i] for i in test_idx]

    X_train_precomputed = X_known_full.loc[train_mol_ids, train_mol_ids].values
    X_test_precomputed = X_known_full.loc[test_mol_ids, train_mol_ids].values

    knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric='precomputed')
    knn.fit(X_train_precomputed, y_train_labels_split)

    y_pred_test = knn.predict(X_test_precomputed)

    print("--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---")
    print("Matriz de confusión:")
    print(confusion_matrix(y_test_labels_split, y_pred_test))
    print("\nReporte de clasificación:")
    print(classification_report(y_test_labels_split, y_pred_test, zero_division=0))

    # 3. Predicción para moléculas desconocidas (indexación directa por label, ya únicas).
    if unknown_mol_ids:
        X_unknown_to_train = full_distance_df.loc[unknown_mol_ids, train_mol_ids].values

        y_pred_unknown_proba = knn.predict_proba(X_unknown_to_train)
        class_labels = knn.classes_  # Códigos hexadecimales de color

        color_hex_to_name_map_local = {
            '#008000': 'DarkGreen',
            '#8B7500': 'MustardYellow',
            '#8B0000': 'DarkRed',
            '#555555': 'DarkGray',
            '#0000CD': 'MediumBlue',
        }

        predicted_class_indices = np.argmax(y_pred_unknown_proba, axis=1)
        predicted_hex_colors = [class_labels[idx] for idx in predicted_class_indices]
        predicted_color_names = [color_hex_to_name_map_local.get(h, h) for h in predicted_hex_colors]

        print("\n--- Predicciones para moléculas desconocidas ---")
        proba_df = pd.DataFrame(
            y_pred_unknown_proba,
            columns=[f'{color_hex_to_name_map_local.get(c, c)}_proba' for c in class_labels]
        )
        proba_df.insert(0, 'mol_id', unknown_mol_ids)
        proba_df.insert(1, 'predicted_affinity_color_name', predicted_color_names)
        proba_df.insert(2, 'predicted_affinity_color_hex', predicted_hex_colors)

        display(proba_df)
        return proba_df
    else:
        print("\nNo hay moléculas desconocidas para predecir.")
        return None

Finalmente, se itera sobre cada una de las matrices de distancia normalizadas y se aplica
`perform_knn_classification` a todas, mostrando los resultados de forma secuencial.

In [10]:
distance_matrices = {
    'distance_neg_norm': distance_neg_norm,
    'distance_pos_col_norm': distance_pos_col_norm,
    'distance_mix_50_50_norm': distance_mix_50_50_norm,
    'distance_mix_30_70_norm': distance_mix_30_70_norm,
    'distance_mix_70_30_norm': distance_mix_70_30_norm,
}

resultados_clasificacion = {}
for name, df_matrix in distance_matrices.items():
    resultados_clasificacion[name] = perform_knn_classification(
        df_matrix, name, known_mol_ids_present_in_matrix,
        unknown_mol_ids_list, y_known_colors_present_in_matrix
    )


--- Clasificando con distance_neg_norm ---

--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---
Matriz de confusión:
[[1 0 2 3 0]
 [0 0 0 1 0]
 [4 0 3 2 0]
 [2 0 4 1 2]
 [0 0 4 2 0]]

Reporte de clasificación:
              precision    recall  f1-score   support

     #0000CD       0.14      0.17      0.15         6
     #008000       0.00      0.00      0.00         1
     #555555       0.23      0.33      0.27         9
     #8B0000       0.11      0.11      0.11         9
     #8B7500       0.00      0.00      0.00         6

    accuracy                           0.16        31
   macro avg       0.10      0.12      0.11        31
weighted avg       0.13      0.16      0.14        31


--- Predicciones para moléculas desconocidas ---


,mol_id,predicted_affinity_color_name,predicted_affinity_color_hex,MediumBlue_proba,DarkGreen_proba,DarkGray_proba,DarkRed_proba,MustardYellow_proba
0,xyz_newmol0_neg,DarkRed,#8B0000,0.2,0.0,0.0,0.6,0.2
1,xyz_newmol10_neg,DarkGray,#555555,0.2,0.0,0.4,0.4,0.0
2,xyz_newmol11_neg,DarkGray,#555555,0.2,0.0,0.4,0.4,0.0
3,xyz_newmol12_neg,MustardYellow,#8B7500,0.2,0.0,0.0,0.2,0.6
4,xyz_newmol13_neg,MediumBlue,#0000CD,0.2,0.2,0.2,0.2,0.2
5,xyz_newmol14_neg,MustardYellow,#8B7500,0.0,0.0,0.0,0.4,0.6
6,xyz_newmol15_neg,DarkGray,#555555,0.2,0.0,0.4,0.4,0.0
7,xyz_newmol16_neg,MustardYellow,#8B7500,0.2,0.2,0.0,0.2,0.4
8,xyz_newmol17_neg,DarkRed,#8B0000,0.2,0.0,0.2,0.6,0.0
9,xyz_newmol18_neg,DarkGray,#555555,0.2,0.0,0.4,0.0,0.4



--- Clasificando con distance_pos_col_norm ---

--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---
Matriz de confusión:
[[0 0 2 4 0]
 [0 1 0 0 0]
 [0 0 3 5 1]
 [1 0 3 5 0]
 [0 0 1 3 2]]

Reporte de clasificación:
              precision    recall  f1-score   support

     #0000CD       0.00      0.00      0.00         6
     #008000       1.00      1.00      1.00         1
     #555555       0.33      0.33      0.33         9
     #8B0000       0.29      0.56      0.38         9
     #8B7500       0.67      0.33      0.44         6

    accuracy                           0.35        31
   macro avg       0.46      0.44      0.43        31
weighted avg       0.34      0.35      0.33        31


--- Predicciones para moléculas desconocidas ---


,mol_id,predicted_affinity_color_name,predicted_affinity_color_hex,MediumBlue_proba,DarkGreen_proba,DarkGray_proba,DarkRed_proba,MustardYellow_proba
0,xyz_newmol0_neg,DarkGray,#555555,0.0,0.0,0.6,0.2,0.2
1,xyz_newmol10_neg,DarkRed,#8B0000,0.0,0.0,0.2,0.4,0.4
2,xyz_newmol11_neg,DarkGray,#555555,0.0,0.0,0.6,0.4,0.0
3,xyz_newmol12_neg,DarkRed,#8B0000,0.0,0.0,0.0,0.8,0.2
4,xyz_newmol13_neg,DarkGray,#555555,0.2,0.0,0.4,0.0,0.4
5,xyz_newmol14_neg,MustardYellow,#8B7500,0.0,0.0,0.2,0.2,0.6
6,xyz_newmol15_neg,DarkRed,#8B0000,0.0,0.0,0.2,0.6,0.2
7,xyz_newmol16_neg,DarkGreen,#008000,0.0,0.4,0.2,0.2,0.2
8,xyz_newmol17_neg,DarkGreen,#008000,0.0,0.4,0.0,0.4,0.2
9,xyz_newmol18_neg,DarkGray,#555555,0.0,0.0,0.4,0.4,0.2



--- Clasificando con distance_mix_50_50_norm ---

--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---
Matriz de confusión:
[[1 0 2 3 0]
 [0 1 0 0 0]
 [2 0 3 4 0]
 [1 0 4 4 0]
 [0 0 3 3 0]]

Reporte de clasificación:
              precision    recall  f1-score   support

     #0000CD       0.25      0.17      0.20         6
     #008000       1.00      1.00      1.00         1
     #555555       0.25      0.33      0.29         9
     #8B0000       0.29      0.44      0.35         9
     #8B7500       0.00      0.00      0.00         6

    accuracy                           0.29        31
   macro avg       0.36      0.39      0.37        31
weighted avg       0.24      0.29      0.25        31


--- Predicciones para moléculas desconocidas ---


...


--- Clasificando con distance_mix_30_70_norm ---

--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---
Matriz de confusión:
[[0 0 1 5 0]
 [0 1 0 0 0]
 [1 0 1 7 0]
 [0 0 5 4 0]
 [0 0 5 1 0]]

Reporte de clasificación:
              precision    recall  f1-score   support

     #0000CD       0.00      0.00      0.00         6
     #008000       1.00      1.00      1.00         1
     #555555       0.08      0.11      0.10         9
     #8B0000       0.24      0.44      0.31         9
     #8B7500       0.00      0.00      0.00         6

    accuracy                           0.19        31
   macro avg       0.26      0.31      0.28        31
weighted avg       0.12      0.19      0.15        31


--- Predicciones para moléculas desconocidas ---


,mol_id,predicted_affinity_color_name,predicted_affinity_color_hex,MediumBlue_proba,DarkGreen_proba,DarkGray_proba,DarkRed_proba,MustardYellow_proba
0,xyz_newmol0_neg,DarkRed,#8B0000,0.0,0.0,0.2,0.6,0.2
1,xyz_newmol10_neg,DarkRed,#8B0000,0.0,0.0,0.4,0.6,0.0
2,xyz_newmol11_neg,DarkGray,#555555,0.0,0.0,0.6,0.4,0.0
3,xyz_newmol12_neg,DarkRed,#8B0000,0.0,0.0,0.2,0.4,0.4
4,xyz_newmol13_neg,MustardYellow,#8B7500,0.2,0.0,0.2,0.2,0.4
5,xyz_newmol14_neg,DarkRed,#8B0000,0.0,0.0,0.2,0.4,0.4
6,xyz_newmol15_neg,DarkRed,#8B0000,0.0,0.0,0.2,0.8,0.0
7,xyz_newmol16_neg,DarkGreen,#008000,0.0,0.4,0.2,0.2,0.2
8,xyz_newmol17_neg,DarkGreen,#008000,0.0,0.4,0.0,0.4,0.2
9,xyz_newmol18_neg,DarkRed,#8B0000,0.0,0.0,0.4,0.6,0.0



--- Clasificando con distance_mix_70_30_norm ---



--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---
Matriz de confusión:
[[1 0 2 3 0]
 [0 0 0 0 1]
 [2 0 3 4 0]
 [1 0 5 3 0]
 [0 0 2 4 0]]

Reporte de clasificación:
              precision    recall  f1-score   support

     #0000CD       0.25      0.17      0.20         6
     #008000       0.00      0.00      0.00         1
     #555555       0.25      0.33      0.29         9
     #8B0000       0.21      0.33      0.26         9
     #8B7500       0.00      0.00      0.00         6

    accuracy                           0.23        31
   macro avg       0.14      0.17      0.15        31
weighted avg       0.18      0.23      0.20        31


--- Predicciones para moléculas desconocidas ---


,mol_id,predicted_affinity_color_name,predicted_affinity_color_hex,MediumBlue_proba,DarkGreen_proba,DarkGray_proba,DarkRed_proba,MustardYellow_proba
0,xyz_newmol0_neg,DarkRed,#8B0000,0.0,0.0,0.2,0.6,0.2
1,xyz_newmol10_neg,DarkGray,#555555,0.0,0.0,0.6,0.4,0.0
2,xyz_newmol11_neg,DarkGray,#555555,0.0,0.0,0.6,0.4,0.0
3,xyz_newmol12_neg,MustardYellow,#8B7500,0.2,0.0,0.0,0.2,0.6
4,xyz_newmol13_neg,MediumBlue,#0000CD,0.2,0.2,0.2,0.2,0.2
5,xyz_newmol14_neg,DarkRed,#8B0000,0.0,0.0,0.0,0.6,0.4
6,xyz_newmol15_neg,DarkGray,#555555,0.2,0.0,0.6,0.2,0.0
7,xyz_newmol16_neg,DarkRed,#8B0000,0.0,0.2,0.2,0.6,0.0
8,xyz_newmol17_neg,DarkRed,#8B0000,0.0,0.0,0.2,0.6,0.2
9,xyz_newmol18_neg,DarkGray,#555555,0.0,0.0,0.4,0.2,0.4


## 5. K-Nearest Neighbors (KNN) — Regresión

Ahora implementamos un modelo de regresión KNN para predecir los valores de afinidad
exactos, en lugar de categorías de color. Se usa `KNeighborsRegressor` de `sklearn` con
`metric='precomputed'`, evaluado con Error Cuadrático Medio (MSE) y R².

In [11]:
# Valores de afinidad continuos para las moléculas conocidas.
y_known_affinity_values_present_in_matrix = [
    bace_affinity_df[bace_affinity_df['ID_real'].astype(str) == mol_id]['Affinity'].values[0]
    for mol_id in known_mol_ids_present_in_matrix
]

print(f"Longitud de y_known_affinity_values para regresión: {len(y_known_affinity_values_present_in_matrix)}")

Longitud de y_known_affinity_values para regresión: 153


In [12]:
def perform_knn_regression(full_distance_df, matrix_name, known_mol_ids, unknown_mol_ids,
                            y_known_affinity, n_neighbors=5):
    print(f"\n--- Regresión con {matrix_name} ---\n")

    # 1. Sub-matriz de distancias entre moléculas conocidas.
    X_known_full = full_distance_df.loc[known_mol_ids, known_mol_ids]

    # 2. División train/test sobre los índices de las moléculas conocidas.
    known_indices = list(range(len(known_mol_ids)))
    train_idx, test_idx, y_train_affinity_split, y_test_affinity_split = train_test_split(
        known_indices, y_known_affinity, test_size=0.2, random_state=42
    )
    train_mol_ids = [known_mol_ids[i] for i in train_idx]
    test_mol_ids = [known_mol_ids[i] for i in test_idx]

    X_train_precomputed = X_known_full.loc[train_mol_ids, train_mol_ids].values
    X_test_precomputed = X_known_full.loc[test_mol_ids, train_mol_ids].values

    knn_reg = KNeighborsRegressor(n_neighbors=n_neighbors, metric='precomputed')
    knn_reg.fit(X_train_precomputed, y_train_affinity_split)

    y_pred_test_reg = knn_reg.predict(X_test_precomputed)

    print("--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---")
    print(f"Error Cuadrático Medio (MSE): {mean_squared_error(y_test_affinity_split, y_pred_test_reg):.4f}")
    print(f"Coeficiente R-cuadrado (R2): {r2_score(y_test_affinity_split, y_pred_test_reg):.4f}")

    # 3. Predicción para moléculas desconocidas (indexación directa por label, ya únicas).
    if unknown_mol_ids:
        X_unknown_to_train = full_distance_df.loc[unknown_mol_ids, train_mol_ids].values
        y_pred_unknown_reg = knn_reg.predict(X_unknown_to_train)

        print("\n--- Predicciones para moléculas desconocidas ---")
        unknown_predictions_reg_df = pd.DataFrame({
            'mol_id': unknown_mol_ids,
            'predicted_affinity_value': y_pred_unknown_reg,
        })
        display(unknown_predictions_reg_df)
        return unknown_predictions_reg_df
    else:
        print("\nNo hay moléculas desconocidas para predecir.")
        return None

Finalmente, iteramos a través de cada una de las matrices de distancia normalizadas y
aplicamos `perform_knn_regression`.

In [13]:
resultados_regresion = {}
for name, df_matrix in distance_matrices.items():
    resultados_regresion[name] = perform_knn_regression(
        df_matrix, name, known_mol_ids_present_in_matrix,
        unknown_mol_ids_list, y_known_affinity_values_present_in_matrix
    )


--- Regresión con distance_neg_norm ---

--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---
Error Cuadrático Medio (MSE): 30.8182
Coeficiente R-cuadrado (R2): -17.2044

--- Predicciones para moléculas desconocidas ---


,mol_id,predicted_affinity_value
0,xyz_newmol0_neg,0.50070
1,xyz_newmol10_neg,0.10980
2,xyz_newmol11_neg,0.38380
3,xyz_newmol12_neg,1.37520
4,xyz_newmol13_neg,3.55550
5,xyz_newmol14_neg,8.42990
6,xyz_newmol15_neg,8.08680
7,xyz_newmol16_neg,3.26230
8,xyz_newmol17_neg,0.26750
9,xyz_newmol18_neg,0.51740



--- Regresión con distance_pos_col_norm ---



--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---
Error Cuadrático Medio (MSE): 17.7099
Coeficiente R-cuadrado (R2): -9.4613

--- Predicciones para moléculas desconocidas ---


,mol_id,predicted_affinity_value
0,xyz_newmol0_neg,1.2436
1,xyz_newmol10_neg,0.9478
2,xyz_newmol11_neg,0.1536
3,xyz_newmol12_neg,1.4320
4,xyz_newmol13_neg,1.4154
5,xyz_newmol14_neg,1.5296
6,xyz_newmol15_neg,0.6380
7,xyz_newmol16_neg,22.8396
8,xyz_newmol17_neg,14.9736
9,xyz_newmol18_neg,0.1816


...


--- Regresión con distance_mix_30_70_norm ---



--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---
Error Cuadrático Medio (MSE): 19.3983
Coeficiente R-cuadrado (R2): -10.4587

--- Predicciones para moléculas desconocidas ---


,mol_id,predicted_affinity_value
0,xyz_newmol0_neg,1.3120
1,xyz_newmol10_neg,0.1380
2,xyz_newmol11_neg,0.5236
3,xyz_newmol12_neg,1.5420
4,xyz_newmol13_neg,2.8656
5,xyz_newmol14_neg,2.0496
6,xyz_newmol15_neg,0.6778
7,xyz_newmol16_neg,22.8316
8,xyz_newmol17_neg,14.9860
9,xyz_newmol18_neg,0.2034



--- Regresión con distance_mix_70_30_norm ---

--- Evaluación sobre moléculas conocidas (conjunto de prueba) ---
Error Cuadrático Medio (MSE): 17.0890
Coeficiente R-cuadrado (R2): -9.0945

--- Predicciones para moléculas desconocidas ---


,mol_id,predicted_affinity_value
0,xyz_newmol0_neg,1.31200
1,xyz_newmol10_neg,0.10980
2,xyz_newmol11_neg,0.59580
3,xyz_newmol12_neg,2.63000
4,xyz_newmol13_neg,1.60550
5,xyz_newmol14_neg,2.02990
6,xyz_newmol15_neg,0.13480
7,xyz_newmol16_neg,2.75340
8,xyz_newmol17_neg,0.39190
9,xyz_newmol18_neg,0.51740
